In [1]:
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os 

from model.nnModel import Net

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import pickle

In [ ]:
run = wandb.init(
    entity="sanjana-learning",
    project="cat-or-dog",
    config={
        "architecture": "CNN",
        "dataset": "https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset/data",
    },
)

In [ ]:
#overfitting do a validation split 
#check and see how many are being classified correctly and incorrectly
#save multiple models and compare them
#make graphs for each one

In [ ]:
# -------------------------
# 1) Load data
# -------------------------
X_data = np.load("data/catdog_X.npy", allow_pickle=True)
y_data = np.load("data/catdog_y.npy", allow_pickle=True)

# X should become [N, 1, 50, 50] float32
X = torch.tensor(np.array(X_data), dtype=torch.float32)
y = torch.tensor(np.array(y_data), dtype=torch.float32)  # [N,2]
y = y.argmax(dim=1).long()   

# If your y is shape [N,1], flatten to [N]
y = y.view(-1)

# If your X is currently [N, 50, 50], add channel dim -> [N,1,50,50]
if X.ndim == 3:
    X = X.unsqueeze(1)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype, "unique:", torch.unique(y))
assert len(X) == len(y)


In [ ]:
# -------------------------
# 2) Split train/test
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y.numpy()  # keeps class balance similar in train/test
)

np.save("data/catdog_X_test.npy", X_test)
np.save("data/catdog_y_test.npy", y_test)

In [ ]:
# -------------------------
# 3) Setup model + training
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
net = Net().to(device)

optimizer = optim.Adam(net.parameters(), lr=1e-3)
loss_function = nn.CrossEntropyLoss()

batch_size = 100
epochs = 5

In [ ]:
def get_metrics(subset, net, X, y, train_loss, epoch, k):
    net.eval()
    with torch.no_grad():
        X_te = X.to(device)
        logits_te = net(X_te)

        probs_pos = torch.softmax(logits_te, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(logits_te, dim=1).cpu().numpy()
        y_true = y.cpu().numpy()

        cm = confusion_matrix(y_true, preds)
        acc = accuracy_score(y_true, preds)
        prec = precision_score(y_true, preds, zero_division=0)
        rec = recall_score(y_true, preds, zero_division=0)
        f1 = f1_score(y_true, preds, zero_division=0)

        try:
            auc = roc_auc_score(y_true, probs_pos)
        except ValueError:
            auc = float("nan")

    return {
        "subset": subset,
        "fold": k,
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "cm":cm
    }

In [ ]:
def log_param_stats_wandb(net, k, epoch):
    payload = {"epoch": epoch + 1, "fold": k}

    with torch.no_grad():
        for name, p in net.named_parameters():
            if not p.requires_grad:
                continue
            x = p.detach().float().view(-1).cpu().numpy()

            base = f"fold_{k}/params/{name}"
            payload[f"{base}/mean"] = float(x.mean())
            payload[f"{base}/std"]  = float(x.std())
            payload[f"{base}/l2"]   = float(np.linalg.norm(x))

    wandb.log(payload, step=epoch + 1)  # step=epoch is fine since fold is in the key

In [ ]:
# -------------------------
# 5) Training loop
# -------------------------

#i am currently passing the entire dataset through the model each epoch 
# would it be better to do k-fold cross validation?
validation_scores = []
kf = KFold(n_splits=5,random_state=42,shuffle=True) 
k = 0

for train_index, val_index in kf.split(X_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]
    k+=1
    for epoch in range(epochs):
        net.train() 
        perm = torch.randperm(len(X_train_fold))
        X_train_shuf = X_train_fold[perm]
        y_train_shuf = y_train_fold[perm]

        running_loss = 0.0

        for i in range(0, len(X_train_shuf), batch_size):
            batch_X = X_train_shuf[i:i+batch_size].to(device) #is to(device) necessary?
            batch_y = y_train_shuf[i:i+batch_size].to(device)
      
            optimizer.zero_grad(set_to_none=True)
            logits = net(batch_X)                     # logits shape [B,2]
            loss = loss_function(logits, batch_y)     # batch_y shape [B]
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * batch_X.size(0)

        train_loss = running_loss / len(X_train_shuf)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}")

        # -------------------------
        # 6) Evaluate once per epoch
        # -------------------------
        # log eval metrics (dicts)
        train_log = get_metrics("train", net, X_train_fold, y_train_fold, train_loss, epoch, k)
        val_log   = get_metrics("val",   net, X_val_fold,   y_val_fold,   train_loss, epoch, k)

        #validation_scores.append(train_log).append(val_log)
        print(train_log)
        print(val_log)
        validation_scores.append(train_log)
        validation_scores.append(val_log)
        torch.save(net.state_dict(), f"model/catdog_cnn_model_k{k}e{epoch+1}.pth")

    # evidence of improvement:
    # train_loss should decrease across epochs (printed each epoch)
    # Validation metrics (accuracy, precision, recall, f1, auc) should improve


In [ ]:
# -------------------------
# 6) Export Metrics
# -------------------------
file_path = 'data/validationScores.pkl'

if not os.path.exists(file_path):
    with open(file_path, 'wb') as file:
        pickle.dump(validation_scores, file)
        print(f"List successfully pickled to {file_path}")
else: 
    print(f"file exists, choose new file name or delete existing file")

In [ ]:
#TODO 
#save model after each epoch/K
#assess on validation, train and test 
#updated the logs 